# Gemma-4 QLoRA SFT (ARCHITECTURE.md §2.4, §5)

Runs the repo's SFT stage on a free-tier T4/P100 session. Requirements:

1. Runtime → **GPU (T4 or P100)** — no GPU exits early.
2. `+ Add-ons → Secrets` → add **GH_TOKEN** (GitHub PAT with `repo` scope) so the private repo can be cloned.
3. Kaggle account must have **Gemma 4 accepted** on the model page.
4. 12 h/session limit (§5): checkpoints save every 100 steps; re-run with `--resume auto` to continue.

In [ ]:
import os, pathlib, subprocess, sys

assert os.environ.get("KAGGLE_KERNEL_RUN_TYPE") or os.environ.get("KAGGLE_GPU_RUNNING_TYPE"), (
    "Turn on the GPU runtime first (T4/P100) — ARCHITECTURE §5"
)
token = os.environ.get("GH_TOKEN")
assert token, "Add the GH_TOKEN secret (GitHub PAT, repo scope) before running"

repo_dir = pathlib.Path("gemma4-coding-agent")
if not repo_dir.exists():
    subprocess.run(
        f"git clone https://{token}@github.com/Ashura-asura/gemma4-coding-agent.git",
        shell=True, check=True,
    )
os.chdir(repo_dir.resolve())
subprocess.run(f'"{sys.executable}" -m pip install -q -e ".[train]"', shell=True, check=True)
print("repo ready at", pathlib.Path.cwd())

In [ ]:
# pre-flight (CPU): collator tests + encode a trajectory against the real Gemma template
!python -m pytest train/tests -q
!python -m train.sft.trainer --config configs/sft_config.yaml --dry-run

In [ ]:
# QLoRA SFT — base model resolves to kagglehub google/gemma-4/transformers/gemma-4-12b on Kaggle
!bash scripts/run_sft.sh --resume auto

In [ ]:
import subprocess

!du -sh checkpoints/sft
subprocess.run(
    f'"{sys.executable}" -m zipfile -c /kaggle/working/sft_adapter.zip checkpoints/sft',
    shell=True, check=True,
)
print("download sft_adapter.zip from /kaggle/working after the run")